# RAG evaluator
- [x] Metrics -> on Retriever component
- [x] Generator component -> with LLM-based judge
- [x] evaluation dataset
- [x] llm main style evaluation
- [x] RAGAS https://medium.com/data-science/evaluating-rag-applications-with-ragas-81d67b0ee31a
- [x] better source data on detailed project and information
- [ ] better golden dataset
- [ ] Dashboard - Once Stable after all the other RAG part


In [1]:
import json
import os
from pathlib import Path
import sys

# Add project root to path so we can import from utils
# In Jupyter notebooks, __file__ doesn't exist, so we use os.getcwd() and check for utils/
current_dir = Path(os.getcwd())
if (current_dir / 'utils').exists():
    # Already at project root
    project_root = current_dir
elif (current_dir.parent / 'utils').exists():
    # We're in src/, go up one level
    project_root = current_dir.parent
else:
    # Fallback: assume we're in src/ and go up one level
    project_root = current_dir.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.base_models import TestQuestion

In [2]:
import tqdm


TEST_QUESTIONS_FILE = "../evaluation/eval_data.jsonl"

def load_test_questions() -> list[TestQuestion]:
    """
    Load test questions from a JSONL file
    """
    with open(TEST_QUESTIONS_FILE, "r", encoding="utf-8") as f:
        tests = []
        for line in f:
            data = json.loads(line.strip()) 
            tests.append(TestQuestion(**data))
        print("Loaded {} test questions".format(len(tests)))
    return tests

In [3]:
tests = load_test_questions()

Loaded 50 test questions


In [4]:
tests[0]
print(tests[0].question)
print(tests[0].ground_truth)
print(tests[0].category)
print(tests[0].keywords)

How many teams benefited from Beiji’s n8n rollout and what was the financial impact?
More than 10 teams used the automation workflows, saving about 10,000 SGD per month.
impact
['n8n', '10+ teams', '10k SGD', 'automation']


In [5]:
from collections import Counter
count = Counter([t.category for t in tests])
count

Counter({'ai_engineering': 10,
         'achievement': 5,
         'engineering': 5,
         'personality': 4,
         'platform_engineering': 3,
         'rag_skill': 3,
         'lifestyle': 3,
         'skills': 3,
         'multi_hop': 3,
         'impact': 2,
         'timeline': 2,
         'project_experience': 2,
         'frontend': 1,
         'education': 1,
         'personal_profile': 1,
         'full_stack_ai': 1,
         'future': 1})

In [6]:
from rag_retrieval import fetch_context, generate_answer, rewrite_query

rewrite_query = rewrite_query(tests[0].question)
retrieval_results = fetch_context(tests[0].question)

print("Retrieval results:")
print(retrieval_results)

print("original query:")
print(tests[0].question)

print("rewritten query:")
print(rewrite_query)



Rewritten query: Number of teams benefited from Beiji n8n rollout and the financial impact of the rollout
Retrieval results:
[Document(id='abb554eb-59a7-41ed-8644-3116eaa46ad7', metadata={'moddate': "D:20260122073854Z00'00'", 'total_pages': 1, 'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creationdate': "D:20260122073854Z00'00'", 'doc_type': 'data', 'title': 'Resume_LI_BEIJI', 'author': 'beiji M4', 'page': 0, 'creator': 'Word', 'page_label': '1', 'source': 'Resume_LI_BEIJI.pdf'}, page_content='Software Engineer Aug 2023 - Present United Overseas Bank  • Led adoption and production rollout of n8n, empowering 10+ teams to build customized automation workflow or agentic solution with 10k SGD saved per month. • Designed and implemented Prompt Template Hub(React & Spring Boot) with Bitbucket integration, providing an interactive dashboard to all domains with version control and access control for 200+ templates. • Built in-house Developer Portal(Angular & Spring Boot) a

### Generator Part Evaluation

In [7]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from utils.base_models import RetrievalLLMEval
from rag_retrieval import generate_answer
from utils.prompts import SYSTEM_PROMPT_EVALUATOR_GENERATOR


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0);

def evaluate_response(test_question: TestQuestion) -> RetrievalLLMEval:
    """
    Evaluate the LLM-Response based on the retrieval results, not on the retrieval results based on the question
    """
    # get the context
    generated_answer, retrieval_results = generate_answer(test_question.question)

    # parse the messages
    system_messages = [SystemMessage(
        content=("You are an expert evaluator assessing the quality of answers. Evaluate the generated answer by comparing it to the reference answer. Only give 5/5 scores for perfect answers."
                 ))]
    user_messages = [HumanMessage(content=SYSTEM_PROMPT_EVALUATOR_GENERATOR.format(question=test_question.question, generated_answer=generated_answer, ground_truth=test_question.ground_truth))]
    messages = system_messages + user_messages

    structured_llm = llm.with_structured_output(RetrievalLLMEval)
    response_LLM_eval = structured_llm.invoke(messages)

    return response_LLM_eval
    

In [8]:
result = evaluate_response(tests[0])

# Check if it's a BaseModel/RetrievalLLMEval instance
print("Type:", type(result))
print("Is RetrievalLLMEval?", isinstance(result, RetrievalLLMEval))
print("\nResult object:")
print(result)
print("\nAccess attributes:")
print(f"accuracy: {result.accuracy}")
print(f"relevance: {result.relevance}")
print(f"completeness: {result.completeness}")
print(f"score: {result.score}")
print(f"confidence: {result.confidence}")
print(f"feedback: {result.feedback}")

Rewritten query: Number of teams benefiting from Beiji’s n8n rollout and its financial impact
Reranked order: [2, 1, 5, 4, 3, 8, 6, 7, 10, 9]
Generated answer with model: gpt-4.1-mini
Type: <class 'utils.base_models.RetrievalLLMEval'>
Is RetrievalLLMEval? True

Result object:
accuracy=4.0 relevance=3.0 completeness=4.0 confidence=0.9 feedback='The retrieval results are mostly correct and relevant, but could be slightly more precise in wording.' score=3.75

Access attributes:
accuracy: 4.0
relevance: 3.0
completeness: 4.0
score: 3.75
confidence: 0.9
feedback: The retrieval results are mostly correct and relevant, but could be slightly more precise in wording.


In [ ]:
def evaluate_LLM(tests: list[TestQuestion]) -> RetrievalLLMEval:
    """
    Evaluate all the tests
    """
    results = []  
    for test in tqdm(tests, desc="Evaluating tests", unit="test"):
        results.append(evaluate_response(test))
    evaluation_result = RetrievalLLMEval(
        accuracy=sum([result.accuracy for result in results]) / len(results),
        relevance=sum([result.relevance for result in results]) / len(results),
        completeness=sum([result.completeness for result in results]) / len(results),
        score=sum([result.score for result in results]) / len(results),
        confidence=sum([result.confidence for result in results]) / len(results),
        feedback="This is the average of all the tests",
    )
    
    return evaluation_result

eval_result_LLM = evaluate_LLM(tests)

Rewritten query: Number of teams benefited from Beiji's n8n rollout and the financial impact of the rollout
Reranked order: [2, 1, 6, 4, 9, 8, 3, 5, 7]
Generated answer with model: gpt-4.1-mini
Rewritten query: Which two internal platforms did Beijing build to improve developer productivity?
Reranked order: [2, 1, 5, 11, 6, 4, 10, 8, 9, 3, 7, 12]
Generated answer with model: gpt-4.1-mini
Rewritten query: What access control problem does the Prompt Template Hub solve?
Reranked order: [4, 3, 1, 9, 2, 5, 6, 7, 8]
Generated answer with model: gpt-4.1-mini
Rewritten query: Which project of Beiji is being scaled by an enterprise Generative AI team?
Reranked order: [1, 2, 5, 6, 3, 7, 8, 4, 9, 10]
Generated answer with model: gpt-4.1-mini
Rewritten query: What retrieval techniques improved the Best Price Notification Agent's performance in information retrieval?
Reranked order: [2, 1, 4, 5, 3, 8, 6, 7]
Generated answer with model: gpt-4.1-mini
Rewritten query: Accuracy improvements of Beiji’s 

In [10]:
print("\nAverage of all the tests from LLM evaluation on LLM answer:")
print(f"accuracy: {eval_result_LLM.accuracy}")
print(f"relevance: {eval_result_LLM.relevance}")
print(f"completeness: {eval_result_LLM.completeness}")
print(f"score: {eval_result_LLM.score}")
print(f"confidence: {eval_result_LLM.confidence}")


Average of all the tests from LLM evaluation on LLM answer:
accuracy: 4.08
relevance: 3.54
completeness: 4.22
score: 4.02
confidence: 0.888


### The metric based evals on RAG retrieval result
- MRR
- Keyword Coverage

In [11]:
def evaluate_mrr(keyword:str, retrieval_results:list) -> float:
    """
    Evaluate the MRR of the retrieval results,
    mrr = 1 -> first result contains the keyword
    mrr = 0.5 -> second result contains the keyword
    mrr = 0 -> no result contains the keyword
    """
    keyword = keyword.lower();
    for rank, result in enumerate(retrieval_results, start=1):
        if keyword in result.page_content.lower():
            return 1/rank
    return 0

In [12]:
from utils.base_models import RetrievalEval


def evaluate_retrieval(test: TestQuestion) -> RetrievalEval:
    """
    Evaluate the retrieval results
    """

    retrieved_docs = fetch_context(test.question)
    mrr_scores = [evaluate_mrr(keyword, retrieved_docs) for keyword in test.keywords]# each keyword need to be calculated separately, so a list of scores
    avg_mrr = sum(mrr_scores) / len(mrr_scores) if mrr_scores else 0.0

    # Calculate keyword coverage
    keywords_found = sum(1 for score in mrr_scores if score > 0)
    total_keywords = len(test.keywords)
    keyword_coverage = (keywords_found / total_keywords * 100) if total_keywords > 0 else 0.0

    return RetrievalEval(
        MRR=avg_mrr,
        keyword_coverage=keyword_coverage,
    )

def evaluate_all(tests: list[TestQuestion]) -> RetrievalEval:
    """
    Evaluate all the tests
    """
    results = []
    for test in tests:
        results.append(evaluate_retrieval(test))
    
    mrr_final = sum(result.MRR for result in results) / len(results)
    keyword_coverage_final = sum(result.keyword_coverage for result in results) / len(results)

    return RetrievalEval(
        MRR=format(mrr_final, ".2f"),
        keyword_coverage=format(keyword_coverage_final, ".2f"),
    )


In [13]:
eval_result_retrieval = evaluate_all(tests)
print("\nAverage of all the tests from retrieval evaluation:")
print(f"MRR: {eval_result_retrieval.MRR}")
print(f"Keyword Coverage: {eval_result_retrieval.keyword_coverage}% ")


Average of all the tests from retrieval evaluation:
MRR: 0.62
Keyword Coverage: 85.93% 


### RAGAS
https://medium.com/data-science/evaluating-rag-applications-with-ragas-81d67b0ee31a

In [14]:
from datasets import Dataset

questions = [test.question for test in tests]
ground_truths = [test.ground_truth for test in tests]
contexts = []
answers = []

"""TODO: reuse the context and generated answer"""
for test in tests:
    test_answer, test_context = generate_answer(test.question)
    answers.append(test_answer)
    # each query have multiple context documents
    contexts.append([doc.page_content for doc in test_context])

dataset = Dataset.from_dict({
    "question": questions,
    "reference": ground_truths,
    "contexts": contexts,
    "answer": answers,
})

Rewritten query: Number of teams benefited from Beiji's n8n rollout and its financial impact details
Reranked order: [2, 1, 10, 4, 6, 9, 3, 5, 8, 7]
Generated answer with model: gpt-4.1-mini
Rewritten query: Which two internal platforms did Beiji build to improve developer productivity?
Reranked order: [2, 1, 5, 4, 6, 9, 3, 8, 7]
Generated answer with model: gpt-4.1-mini
Rewritten query: What access control problem did the Prompt Template Hub address?
Reranked order: [4, 3, 1, 6, 2, 9, 5, 7, 8]
Generated answer with model: gpt-4.1-mini
Rewritten query: Which project of Beiji is being scaled by an enterprise Generative AI team?
Reranked order: [1, 2, 3, 5, 6, 4, 7, 8, 9, 10]
Generated answer with model: gpt-4.1-mini
Rewritten query: retrieval techniques that improved Best Price Notification Agent performance
Reranked order: [4, 2, 3, 1, 5, 9, 6, 7, 8]
Generated answer with model: gpt-4.1-mini
Rewritten query: Accuracy improvements of Beiji’s fine-tuned model compared to GPT-4o and human

In [16]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)

from rag_ingestion import embeddings

"""
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
is because only have 1 LLM, and RAGAS is expecting 3 by default, but 1 still works 
"""

result = evaluate(
    dataset = dataset, 
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    embeddings=embeddings,
    llm=llm,
)

df = result.to_pandas()

/var/folders/q9/87xq5q2d0sv9vrwx1n47nhvc0000gn/T/ipykernel_29615/4164449180.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/var/folders/q9/87xq5q2d0sv9vrwx1n47nhvc0000gn/T/ipykernel_29615/4164449180.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/var/folders/q9/87xq5q2d0sv9vrwx1n47nhvc0000gn/T/ipykernel_29615/4164449180.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
/var/fo

Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[12]: TimeoutError()
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[28]: TimeoutError()
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM return

In [17]:
df.head()

,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,How many teams benefited from Beiji’s n8n roll...,[Software Engineer Aug 2023 - Present United O...,Beiji's n8n rollout benefited over 10 teams an...,More than 10 teams used the automation workflo...,1.000000,1.0,1.000,0.760874
1,Which two internal platforms did Beiji build t...,[for 200+ templates. • Built in-house Develope...,Beiji built two internal platforms to improve ...,He built an in-house Developer Portal and work...,1.000000,1.0,1.000,0.911166
2,What access control problem did the Prompt Tem...,[for 200+ templates. • Built in-house Develope...,The Prompt Template Hub designed and implement...,It provided version control and access control...,0.416667,1.0,0.500,0.718271
3,Which project of Beiji’s is being scaled by an...,[• Engineered a backend service (Python FastAP...,Beiji's project involving the backend service ...,The LLM-Based Bitbucket Code Reviewer is being...,NaN,0.0,1.000,0.597301
4,What retrieval techniques improved the Best Pr...,[•\tDeveloped a dynamic dashboard (React) disp...,The Best Price Notification Agent's performanc...,"Query rewriting, reranking, and optimized embe...",0.500000,1.0,0.875,0.858720


In [18]:
print("contect precision: ", format(df["context_precision"].mean(), ".2f"))
print("contect recall: ", format(df["context_recall"].mean(), ".2f"))
print("faithfulness: ", format(df["faithfulness"].mean(), ".2f"))
print("answer relevancy: ", format(df["answer_relevancy"].mean(), ".2f"))

contect precision:  0.66
contect recall:  0.95
faithfulness:  0.89
answer relevancy:  0.85
